<a href="https://colab.research.google.com/github/PovSobek/Manipuladores/blob/main/E01_forward_kinematics_with_POE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Exercice 01. Forward kinematics with POE


In this exercise you will learn basic skill for working with screws, and particulary with the the definition of Product of Exponentials (POE)


> $e^{\tilde{\zeta_{1}}\theta_{1}}$




The product of exponentials (POE) method is a robotics convention for mapping the links of a spatial kinematic chain. It is an alternative to Denavit–Hartenberg parameterization. While the latter method uses the minimal number of parameters to represent joint motions, the former method has a number of advantages: uniform treatment of prismatic and revolute joints, definition of only two reference frames, and an easy geometric interpretation from the use of screw axes for each joint.

For a video explanation of POE formula, you can visit:

   

*   https://youtu.be/hE_Duih_7JE
*   https://youtu.be/27jUrkFdyks


  

---



After review the basic concepts, you will have to solve the Forward Kinematics
of your selected robot at the section "Example with my robot"

```
# Exercise:
After review the basic concepts, you will have to solve the Forward Kinematics
of your selected robot at the section "Example with my robot"
```




# Skew operator. The key for product of exponentials manipulation

"axis2skew" Generate a skew symmetric matrix from a vector (axis).
Use in SO(3).

	r = axis2skew(w)

Returns a skew symmetric matrix r 3x3 from the vector 3x1 $\mathbf{a} = [a_1, a_2, a_3]$.


\begin{align}
\mathbf{R}=\begin{bmatrix}
0 & -a_3 & a_2\\
a_3 & 0 & -a_1\\
-a_2 & a_1 & 0
\end{bmatrix}
\end{align}


It is useful to transform a vector cross product to a matrix product, in the case of 3D vectors.

"Cross = a X b" is the same as "Cross = axis2skew(a) * b"

In [ ]:
#from re import U
import math #para poder usar math.pi
import numpy as np


def axis2skeworig(u):
  # Generate a skew symmetric matrix from a vector (axis u)
  return np.matrix((
      (0.0, -u[2], u[1]),
      (u[2], 0.0, -u[0]),
      (-u[1], u[0], 0.0),
      ))

def axis2skew(u):
  # Generate a skew symmetric matrix from a vector (axis u)
  return np.array([
      [0.0, -u[2], u[1]],
      [u[2], 0.0, -u[0]],
      [-u[1], u[0], 0.0]
      ])


a = np.array([2, 0, 1])
b = np.array([1, -1, 3])
print("cross product: ", np.cross(a, b))
print("--"*20, "ahora con la matrix skew")
m = axis2skew(a)
print("m matrix: \n", m)
print("skew operator1: @ ", m @ b) #opcion preferente
print("skew operator2: matmul", np.matmul(m, b)) #deprecated
print("skew operator3: .dot", m.dot(b))
print("skew operator4: * \n", m*b) #ojo, esta sintaxis no es equivalente. Es element wise multipy

#Exponential screw

# 2D version. Screw Rotation (ONLY orientation)
\begin{equation}
e^{\tilde{u}\theta}=\left[\mathbf{I}+\mathbf{\tilde{u}}\sin\theta+(\mathbf{\tilde{u}})^2(1-\cos\theta)\right]
\end{equation}

Dada una matriz de rotación $\mathbf{R}\in SO(3)$, existe un vector unitario $\mathbf{u}\in \mathbb{R}^3$ y un escalar $\theta \in \mathbb{R}$, tal que
$\mathbf{R}=e^{\tilde{u}\theta}$


# Coordenadas exponenciales del movimiento de un cuerpo. Rigid Body Motion



*   $\omega=0$.  En este caso el desarrollo de la exponencial se obtiene que $e^{\mathbf{T}\theta}=I+\mathbf{T}\theta$

 \begin{equation}
    \mathbf{A}=e^{\mathbf{T}\theta}=\begin{bmatrix}
    I & v\theta\\
    0 & 1\\
    \end{bmatrix}
    \hspace{1.5cm} \omega = 0
    \end{equation}

*   $\omega \neq 0$. En [Murray94] se demuestra que en el caso general:

\begin{equation}
    \mathbf{A}=e^{\mathbf{T}\theta}=\begin{bmatrix}
    e^{\tilde{u}\theta} & \left(I-e^{\tilde{u}\theta}\right)\left(\omega \times v \right)+ \omega \omega^T v \theta  \\
    0 & 1\\
    \end{bmatrix}
    \hspace{1.5cm} \omega \neq 0
    \end{equation}



In [ ]:
def expAxAng(AxAng):
  '''
  RODRIGUES FORMULA - Exponential Matrix of a Rigid Body ORIENTATION
  AxAng is a 4x1  [x y z theta] == [W theta] (1x4)
  '''
  axis_orig = (AxAng[0], AxAng[1], AxAng[2]) # OJO es una lista no numpy
  #print("axis_orig: ", axis_orig)
  axis = np.array(AxAng[:3]) #asi seria un array numpy
  #print("axis: ", axis)
  u_orig = axis2skeworig(axis_orig)
  u = axis2skew(axis)
  #print("u matrix orig: \n", u_orig, u_orig.dtype)
  #print("u matrix: \n", u, u.dtype)
  theta = AxAng[3]
  I = np.eye(3)
  A_orig = I + u_orig*np.sin(theta)  + u_orig*u_orig*(1-np.cos(theta)) #expresion general de la formula de Rodrigues
  A = I + u*np.sin(theta)  + u@u*(1-np.cos(theta)) #expresion general de la formula de Rodrigues
  #print ("u_orig*np.sin(theta).  : \n", u_orig*np.sin(theta))
  #print ("u*np.sin(theta).  :\n ", u*np.sin(theta))
  #print ("u_orig*u_orig  : \n", u_orig*u_orig)
  #print ("u*u  :\n ", u@u)
  #print ("u_orig*u_orig*(1-np.cos(theta)  : \n", u_orig*u_orig*(1-np.cos(theta)))
  #print ("u*u*(1-np.cos(theta)  :\n ", u@u*(1-np.cos(theta)))
  #print ("A_orig: \n", A_orig)
  #print ("A: \n", A)

  return A

# comprobacion de la funcion
gamma = math.pi/4
w = [0,0,1,math.pi/4]
R_AxAng = expAxAng(w)
print ("R_AxAng =  \n", R_AxAng)

In [ ]:
def expScrew(TwMag):
  # Matrix Exponential of a  Rigid Body Motion by SCREW movement
  # "Twist-Mangitude" (TwMag) [xi; theta] (7x1)
  # Returns a homogeneous "H" matrix (4x4)
  v = np.array([TwMag[0], TwMag[1], TwMag[2]]) #TODO cambiar a slicing
  w = np.array([TwMag[3], TwMag[4], TwMag[5]]) #TODO cambiar a slicing
  theta = TwMag[6]
  if np.linalg.norm(w) == 0:
    R = np.eye(3)
    P = v.dot(theta)
    print ("P: \n",P)
  else:
    R = expAxAng(np.append(w, theta))
    I = np.eye(3)
    #P_aux = np.matmul((I-R),(np.cross(w,v))) #OJO, falta el término + np.transpose(w*np.transpose(w)*v*theta)
    P_aux = np.matmul((I-R),(np.cross(w,v)))+ np.transpose(w*np.transpose(w)*v*theta)
    print ("P_aux \n", P_aux)
    P_aux2 = (I-R)@(np.cross(w,v)) #OJO, falta el término
    print ("P_aux2 \n", P_aux2)
    #P = np.array([P_aux[0,0], P_aux[0,1], P_aux[0,2]])
    P = np.array([P_aux[0], P_aux[1], P_aux[2]])
    #print ("P = ", P)

  return np.array((
      (R[0,0], R[0,1], R[0,2], P[0]),
      (R[1,0], R[1,1], R[1,2], P[1]),
      (R[2,0], R[2,1], R[2,2], P[2]),
      (0, 0, 0, 1),
      )) #TODO mejor hacerlo con la funcion Hmatrix


tw = np.array([0.53, -0.26, 0.8, 0., 0., 0., 14.96])
print ("tw :\n", tw, tw.dtype)
tw2 = np.array([0., 0., 0., 1.0, 0., 0., math.pi/2])
test = expScrew(tw)
print("test (norm == 0):  \n", test)
test2 = expScrew(tw2)
print("test2 (else):  \n = ", test2)
tw3 = np.array([0., -0., 0., 0., 0., 1., 0.])
print ("tw3 :\n", tw3, tw3.dtype)
test3 = expScrew(tw3)
print("test3 :  \n = ", test3)


Exercice 2.3.6: homogeneous rotation.
(es el mismo que el 2.2.6, pero con screws)

Transform a vector $r^{T}(3,2,1)$ expressed in coordinates of the T(UVW) system to its
expression $r^{S}$ in coordinates of the reference system S(XYZ). The system T(UVW) is
rotated by an angle (γ = π/4) about the axis “Z”

In [ ]:
gamma = math.pi/4
w = [0,0,1,math.pi/4]
R_AxAng = expAxAng(w)
print (R_AxAng)
#rt = [3,2,1]
rt = np.array([3, 2, 1])
#rs = R_AxAng.dot(rt)
rs = R_AxAng@rt
print ("rs = ", rs)

Exercice 2.3.7. (es el mismo que el 2.2.7, pero con screws)

Transform a vector r(−3,4,−11) expressed in coordinates of the T(UVW) system to its expression in coordinates of the reference system S(XYZ) (Figure 2.7). The system T(UVW) is rotated “π/2” on the axis OX and then translated by a vector p(8,−4,12), concerning S(XYZ).

\begin{equation}
r^{s}= H_{ST}\cdot r^{T} = e^{\tilde{\zeta_{1}}\theta_{1}}e^{\tilde{\zeta_{2}}\theta_{2}}H_{ST}(0)\cdot r^{T}
\end{equation}

In [ ]:
#rt = np.array([[-3], [4], [-11], [1]]) # vector r columna
rt = np.array([-3, 4, -11, 1]) # vector r
# definicion de la traslacion
#ps = np.array([[8], [-4], [12]])
ps = np.array([8, -4, 12])
theta1 = np.linalg.norm(ps)
omega1 = ps/theta1
aux = np.array([0, 0, 0])
xi1 = np.concatenate((omega1, aux), axis=0)
#print (xi1)
# definicion de la rotacion
theta2 = math.pi/2
#omega2 = np.array([[1], [0], [0]])
omega2 = np.array([1, 0, 0])
xi2 = np.concatenate((aux, omega2), axis=0)
#print (xi2)
#aux3 = np.array
aux2 = np.append(xi1,theta1)
aux3 = np.append(xi2,theta2)
Et1 = expScrew(aux2)
print ("Et1 = ", Et1)
Et2 = expScrew(aux3)
print ("Et2 = ", Et2)
Hst0 = np.eye(4)
#rs = Et1 * Et2 * Hst0 * rt
rs = Et1 @ Et2 @ Hst0 @ rt
print ("rs = ", rs)

# Functions for forwad kinematics with POE

In [ ]:
'''
joint2twist gets de TWIST from the Joint AXIS and a POINT on that axis
where the TWIST "xi" (6x1) has the two components v (3x1) and w (3x1)
From AXIS (3x1) & a POINT (3X1) on that axis AT THE REFERENCE POSITION.
It is also necessary to indicate the type of JOINTTYPE ('rot' or 'tra')
for the function to work with both ROTATION & TRANSLATION movements.
'''

def joint2twist(Axis, Point, JointType):
  # los parametros son LISTAS, pero devuelve un array numpy
  if JointType == 'rot':
    aux = -np.cross(Axis, Point)
    twist = np.concatenate((aux, Axis), axis=0)
  elif JointType == 'tra':
    aux2 = [0, 0, 0]
    twist = np.concatenate((Axis, aux2), axis=0)
  else:
    twist = np.array([0, 0, 0, 0, 0, 0])
  return twist


Axis = [1,0,0] #es un LISTA, no un array numpy
Point = [0, 0, 0]
test= joint2twist(Axis, Point, 'rot')
print( "resultado: ", test )

#Example 4DOF Cilindrical Robot

Compare this result with the DM method

![texto alternativo](https://medicalrobotics.umh.es/files/2022/03/4gdlScrews2.jpg)




\begin{array}{|c|c|c|}
\hline
\textbf{Eslabón} & \mathbf{s} & \mathbf{r_o} \\
 \hline
1 & (0,0,1) & (0,0,0) \\
2 & (0,0,1) & (0,0,0) \\
3 & (0,1,0) & (-a_2,0,l_1) \\
4 & (0,1,0) & (-a_2,l_4,l_1) \\
\hline
\end{array}




In [ ]:
p1 = [0,0,0]
p2 = [0., 0., 0.]
p3 = [-0.1, 0, 0.4]
p4 = [-0.1, 0.2, 0.4]

Axis1 = [0,0,1]
Axis2 = [0,0,1]
Axis3 = [0,1,0]
Axis4 = [0,1,0]

twist1 = joint2twist (Axis1, p1, 'rot')
twist2 = joint2twist (Axis2, p2, 'tra')
twist3 = joint2twist (Axis3, p3, 'tra')
twist4 = joint2twist (Axis4, p4, 'rot')


Hst0 = np.matrix((
      (1, 0, 0, p4[0]),
      (0, 0, -1, p4[1]),
      (0, 1, 0, p4[2]),
      (0, 0, 0, 1),
      ))
magnitude = (0., 0., 1., 0.)
aux1 = np.append(twist1,magnitude[0])
aux2 = np.append(twist2,magnitude[1])
aux3 = np.append(twist3,magnitude[2])
aux4 = np.append(twist4,magnitude[3])

Et1 = expScrew(aux1)
Et2 = expScrew(aux2)
Et3 = expScrew(aux3)
Et4 = expScrew(aux4)

#FK = Et1 * Et2 * Et3 * Et4 * Hst0
FK = Et1 @ Et2 @ Et3 @ Et4 @ Hst0
print ("FK (4dof)=  \n", FK)

# Example of my robot

In [ ]:
# Insert here the code for the FK of your robot